# MULDE Training + GMM on Hiera-L Features

This Colab notebook trains the official MULDE log-density network and the
multiscale GMM aggregation stage on extracted Hiera-L frame features.

It is dataset-configurable. Set `DATASET_NAME` in the configuration cell to:
- `"shanghaitech"` for the original ShanghaiTech feature/label layout.
- `"avenue"` for Avenue features saved under `MULDE/features/avenue_features`.

Repository alignment:
- Reuses the author's `MLPs` and `ScoreOrLogDensityNetwork` from `models.py`.
- Mirrors the denoising score matching objective and GMM aggregation flow from
  the author's `main.py`.
- Adds only dataset/label loading code because the official repo ships a toy
  `dataset.py`, not feature/label loaders for these datasets.

Paper alignment:
- Frame-centric Hiera-L features are 1152-dimensional.
- Hidden layers are `[4096, 4096]` with GELU activations.
- Adam uses `betas=(0.5, 0.9)`.
- Frame-centric learning rate is `1e-4`, matching the CVPR paper text. The repo
  README example uses `4e-5`; this notebook follows the paper.
- Noise range is `[1e-3, 1.0]`, training sigmas are sampled log-uniformly, and
  evaluation uses `L=16` evenly spaced noise levels.
- GMMs use 1, 3, and 5 full-covariance components.

References:
- Paper: https://openaccess.thecvf.com/content/CVPR2024/papers/Micorek_MULDE_Multiscale_Log-Density_Estimation_via_Denoising_Score_Matching_for_Video_CVPR_2024_paper.pdf
- Supplement: https://openaccess.thecvf.com/content/CVPR2024/supplemental/Micorek_MULDE_Multiscale_Log-Density_CVPR_2024_supplemental.pdf
- Official repo: https://github.com/jakubmicorek/MULDE-Multiscale-Log-Density-Estimation-via-Denoising-Score-Matching-for-Video-Anomaly-Detection



## 1. Install Minimal Dependencies

Colab already includes PyTorch in GPU runtimes. This cell installs only the
lightweight packages used by the training/evaluation wrapper.



In [ ]:
import sys
import subprocess

packages = [
    "numpy",
    "pandas",
    "scikit-learn",
    "scipy",
    "tqdm",
    "joblib",
]

subprocess.run([sys.executable, "-m", "pip", "install", "-q", *packages], check=True)
print("Dependency check complete.")



## 2. Mount Google Drive and Import Libraries

All feature inputs and all training/GMM outputs are stored under
`/content/drive/MyDrive/MULDE`.



In [ ]:
import os
import sys
import json
import time
import math
import random
import datetime
import subprocess
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import torch
import torch.optim as optim
from scipy.io import loadmat
from scipy.ndimage import gaussian_filter1d
from sklearn.metrics import roc_auc_score
from sklearn.mixture import GaussianMixture
from torch.utils.data import TensorDataset, DataLoader
from tqdm.auto import tqdm

try:
    from google.colab import drive

    print("Mounting Google Drive at /content/drive ...")
    drive.mount("/content/drive")
except Exception as exc:
    print(f"Google Drive mount skipped or unavailable: {exc}")

print(f"Python: {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")



## 3. Clone and Import the Official MULDE Repository

The official network modules are imported directly from the author's
repository. The notebook saves the cloned commit SHA into the run config for
traceability.



In [ ]:
OFFICIAL_REPO_URL = "https://github.com/jakubmicorek/MULDE-Multiscale-Log-Density-Estimation-via-Denoising-Score-Matching-for-Video-Anomaly-Detection.git"
OFFICIAL_REPO_DIR = Path("/content/MULDE_official")
# Optional reproducibility pin. Leave as None to use the repository default branch.
OFFICIAL_REPO_COMMIT = None

if not OFFICIAL_REPO_DIR.exists():
    subprocess.run(["git", "clone", OFFICIAL_REPO_URL, str(OFFICIAL_REPO_DIR)], check=True)
else:
    print(f"Repository already exists: {OFFICIAL_REPO_DIR}")

if OFFICIAL_REPO_COMMIT is not None:
    subprocess.run(["git", "-C", str(OFFICIAL_REPO_DIR), "checkout", OFFICIAL_REPO_COMMIT], check=True)

repo_sha = subprocess.check_output(
    ["git", "-C", str(OFFICIAL_REPO_DIR), "rev-parse", "HEAD"],
    text=True,
).strip()

sys.path.insert(0, str(OFFICIAL_REPO_DIR))
from models import MLPs, ScoreOrLogDensityNetwork

print(f"Imported official MULDE models from: {OFFICIAL_REPO_DIR}")
print(f"Official repo commit: {repo_sha}")



In [ ]:
# Raw video extraction is not required in this notebook.
# This stage consumes pre-extracted Hiera-L NPZ features plus frame labels.
# If DATASET_NAME="shanghaitech" and its label folder is missing, the
# configuration cell below can optionally extract the ShanghaiTech archive.
print("Skipping raw dataset extraction in the MULDE training notebook.")


## 4. Configuration

Choose the dataset in this cell. The model, DSM objective, GMM stage, and score
export format stay shared; only feature roots, label roots, run directories, and
dataset metadata change.

For Avenue, the expected feature layout is:

`/content/drive/MyDrive/MULDE/features/avenue_features`

with `train/`, `test/`, `manifest_train.csv`, `manifest_test.csv`, and
`train_feature_stats.npz` produced by the Avenue Hiera-L extraction notebook.



In [ ]:
# Dataset selection: "avenue" or "shanghaitech"
DATASET_NAME = "avenue"

# Shared Drive root
DRIVE_ROOT = Path("/content/drive/MyDrive/MULDE")

DATASET_PRESETS = {
    "shanghaitech": {
        "display_name": "ShanghaiTech",
        "feature_root": DRIVE_ROOT / "features" / "hiera_large_16x224_mae_k400_ft_k400_centered_s4",
        "ground_truth_path": Path("/content/shanghaitech/testing/test_frame_mask"),
        "output_run_dir": "shanghaitech_hiera_l_mulde",
        # Only needed if labels have not already been extracted into /content.
        "optional_archive_path": Path("/content/drive/MyDrive/shanghaitech.tar.gz"),
        "optional_archive_extract_to": Path("/content"),
    },
    "avenue": {
        "display_name": "Avenue",
        "feature_root": DRIVE_ROOT / "features" / "avenue_features",
        "ground_truth_path": Path("/content/drive/MyDrive/ground_truth_avenue"),
        "output_run_dir": "avenue_hiera_l_mulde",
        "optional_archive_path": None,
        "optional_archive_extract_to": None,
    },
}

DATASET_KEY = DATASET_NAME.lower().strip()
if DATASET_KEY not in DATASET_PRESETS:
    raise ValueError(f"Unsupported DATASET_NAME={DATASET_NAME!r}. Choose one of: {sorted(DATASET_PRESETS)}")

DATASET_CONFIG = DATASET_PRESETS[DATASET_KEY]
DISPLAY_DATASET_NAME = DATASET_CONFIG["display_name"]

# Input paths
FEATURE_ROOT = Path(DATASET_CONFIG["feature_root"])
GROUND_TRUTH_PATH = Path(DATASET_CONFIG["ground_truth_path"])

TRAIN_MANIFEST_PATH = FEATURE_ROOT / "manifest_train.csv"
TEST_MANIFEST_PATH = FEATURE_ROOT / "manifest_test.csv"
FEATURE_STATS_PATH = FEATURE_ROOT / "train_feature_stats.npz"
TRAIN_FEATURE_DIR = FEATURE_ROOT / "train"
TEST_FEATURE_DIR = FEATURE_ROOT / "test"


def maybe_extract_optional_label_archive() -> None:
    """Extract a dataset archive only when labels are missing and a preset provides one."""
    archive_path = DATASET_CONFIG.get("optional_archive_path")
    extract_to = DATASET_CONFIG.get("optional_archive_extract_to")
    if GROUND_TRUTH_PATH.exists():
        return
    if archive_path is None or extract_to is None:
        return
    archive_path = Path(archive_path)
    extract_to = Path(extract_to)
    if not archive_path.exists():
        print(f"Label archive not found, skipping optional extraction: {archive_path}")
        return
    print(f"Ground-truth path is missing: {GROUND_TRUTH_PATH}")
    print(f"Extracting optional label archive {archive_path} to {extract_to} ...")
    subprocess.run(["tar", "-xzvf", str(archive_path), "-C", str(extract_to)], check=True)


maybe_extract_optional_label_archive()

# Output paths: isolated by dataset name.
RUN_TIMESTAMP = datetime.datetime.now().strftime("%Y_%m_%d_%H_%M_%S")
RUN_TAG = None  # Optional custom folder name; leave None to use timestamp.
RUN_ID = RUN_TAG or RUN_TIMESTAMP
OUTPUT_ROOT = DRIVE_ROOT / "runs" / DATASET_CONFIG["output_run_dir"] / RUN_ID
CHECKPOINT_DIR = OUTPUT_ROOT / "checkpoints"
ARTIFACT_DIR = OUTPUT_ROOT / "artifacts"
LOG_DIR = OUTPUT_ROOT / "logs"

for path in [OUTPUT_ROOT, CHECKPOINT_DIR, ARTIFACT_DIR, LOG_DIR]:
    path.mkdir(parents=True, exist_ok=True)

# Paper-aligned MULDE hyperparameters for frame-centric Hiera-L features
SEED = 42
INPUT_DIM = 1152
UNITS = [4096, 4096]
LR = 1e-4
BATCH_SIZE = 2048
EPOCHS = 845
BETA = 0.1
SIGMA_LOW = 1e-3
SIGMA_HIGH = 1.0
L = 16
ADAM_BETAS = (0.5, 0.9)
GMM_COMPONENTS = [1, 3, 5]
GMM_COVARIANCE = "full"
# Periodic checkpoints are disabled; only the final checkpoint is saved.
CHECKPOINT_EVERY = 845
# Run train/test GMM evaluation every N epochs during training.
PERIODIC_EVAL_EVERY = 5
EVAL_BATCH_SIZE = BATCH_SIZE

# The supplement states that final frame scores are smoothed with a 1D Gaussian
# filter, but does not specify the filter sigma. Keep disabled by default and
# report raw scores.
SMOOTH_SIGMA_FRAMES = None

# Optional restart point. Keep None when switching datasets.
# Example:
RESUME_CHECKPOINT = Path("/content/drive/MyDrive/MULDE/runs/avenue_hiera_l_mulde/Final_Avenue_845/checkpoints/mulde_epoch_0845.pt")
#RESUME_CHECKPOINT = None
if RESUME_CHECKPOINT is not None:
    RESUME_CHECKPOINT = Path(RESUME_CHECKPOINT)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
PIN_MEMORY = DEVICE.type == "cuda"
# Match the author's DataLoader default and avoid duplicating the large feature
# tensor across Colab worker processes.
NUM_WORKERS = 0

CONFIG = {
    "dataset_key": DATASET_KEY,
    "dataset_name": DISPLAY_DATASET_NAME,
    "official_repo_url": OFFICIAL_REPO_URL,
    "official_repo_commit": repo_sha,
    "feature_root": str(FEATURE_ROOT),
    "ground_truth_path": str(GROUND_TRUTH_PATH),
    "train_manifest_path": str(TRAIN_MANIFEST_PATH),
    "test_manifest_path": str(TEST_MANIFEST_PATH),
    "feature_stats_path": str(FEATURE_STATS_PATH),
    "output_root": str(OUTPUT_ROOT),
    "resume_checkpoint": None if RESUME_CHECKPOINT is None else str(RESUME_CHECKPOINT),
    "seed": SEED,
    "input_dim": INPUT_DIM,
    "units": UNITS,
    "lr": LR,
    "batch_size": BATCH_SIZE,
    "epochs": EPOCHS,
    "beta": BETA,
    "sigma_low": SIGMA_LOW,
    "sigma_high": SIGMA_HIGH,
    "L": L,
    "adam_betas": ADAM_BETAS,
    "gmm_components": GMM_COMPONENTS,
    "gmm_covariance": GMM_COVARIANCE,
    "checkpoint_every": CHECKPOINT_EVERY,
    "periodic_eval_every": PERIODIC_EVAL_EVERY,
    "eval_batch_size": EVAL_BATCH_SIZE,
    "smooth_sigma_frames": SMOOTH_SIGMA_FRAMES,
    "device": str(DEVICE),
    "note_lr": "Using 1e-4 from the CVPR paper frame-centric implementation details; repo README example lists 4e-5.",
}

with open(OUTPUT_ROOT / "run_config.json", "w", encoding="utf-8") as f:
    json.dump(CONFIG, f, indent=2)

print(json.dumps(CONFIG, indent=2))


## 5. Reproducibility Helpers

This seeds Python, NumPy, and PyTorch. Exact bitwise reproducibility can still
vary by GPU/runtime, but this fixes the notebook-controlled randomness.



In [ ]:
def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False


seed_everything(SEED)
print(f"Seed set to {SEED}")



## 6. Feature and Label Loading Utilities

These helpers load the NPZ format produced by the Hiera-L feature extraction
notebooks:
- `features`: `[num_frames, 1152]`
- `frame_indices`: `[num_frames]`
- `clip_indices`: `[num_frames, 16]`
- `video_id`, `split`, `num_frames`

Training uses only normal training features. Test labels are mandatory and must
match the number of extracted frame-level features. Label lookup supports both
dataset-level folders and per-row `label_path` values from the test manifest.



In [ ]:
import re

REQUIRED_NPZ_KEYS = {"features", "frame_indices", "clip_indices", "video_id", "split", "num_frames"}
LABEL_EXTENSIONS = {".npy", ".npz", ".mat", ".csv", ".txt"}


def as_python_scalar(value):
    arr = np.asarray(value)
    if arr.shape == ():
        return arr.item()
    return value


def normalize_key(text: str) -> str:
    text = str(text).lower().strip()
    for suffix in [".npy", ".npz", ".mat", ".csv", ".txt", ".avi"]:
        if text.endswith(suffix):
            text = text[: -len(suffix)]
    text = text.replace("-", "_").replace(" ", "_")
    while "__" in text:
        text = text.replace("__", "_")
    return text


def compact_key(text: str) -> str:
    return re.sub(r"[^a-z0-9]+", "", normalize_key(text))


def lookup_keys_for_name(text: str) -> list[str]:
    """Return robust lookup keys for ids like 01, vol01, video_01, 01.avi."""
    normalized = normalize_key(text)
    compact = compact_key(text)
    keys = []

    def add(value):
        if value and value not in keys:
            keys.append(value)

    add(normalized)
    add(compact)

    digit_groups = re.findall(r"\d+", normalized)
    if digit_groups:
        last_number = int(digit_groups[-1])
        add(str(last_number))
        add(f"{last_number:02d}")
        add(f"vol{last_number}")
        add(f"vol{last_number:02d}")
        add(f"video{last_number}")
        add(f"video{last_number:02d}")

    return keys


def validate_feature_npz(path: Path, expected_split: str) -> dict:
    if not path.exists():
        raise FileNotFoundError(f"Missing feature file: {path}")
    data = np.load(path, allow_pickle=True)
    missing = REQUIRED_NPZ_KEYS.difference(data.files)
    if missing:
        raise KeyError(f"{path} is missing required keys: {sorted(missing)}")

    features = data["features"]
    if features.ndim != 2 or features.shape[1] != INPUT_DIM:
        raise ValueError(f"{path}: expected features shape [N, {INPUT_DIM}], got {features.shape}")
    if not np.isfinite(features).all():
        raise ValueError(f"{path}: features contain NaN or Inf")

    frame_indices = data["frame_indices"]
    clip_indices = data["clip_indices"]
    num_frames = int(as_python_scalar(data["num_frames"]))
    video_id = str(as_python_scalar(data["video_id"]))
    split = str(as_python_scalar(data["split"]))

    if split != expected_split:
        raise ValueError(f"{path}: expected split '{expected_split}', found '{split}'")
    if features.shape[0] != num_frames:
        raise ValueError(f"{path}: features length {features.shape[0]} != num_frames {num_frames}")
    if frame_indices.shape[0] != num_frames:
        raise ValueError(f"{path}: frame_indices length {frame_indices.shape[0]} != num_frames {num_frames}")
    if clip_indices.shape != (num_frames, 16):
        raise ValueError(f"{path}: expected clip_indices shape [{num_frames}, 16], got {clip_indices.shape}")

    return {
        "path": path,
        "video_id": video_id,
        "split": split,
        "num_frames": num_frames,
        "features": features.astype(np.float32, copy=False),
        "frame_indices": frame_indices.astype(np.int64, copy=False),
        "clip_indices": clip_indices.astype(np.int64, copy=False),
    }


def load_manifest(path: Path, split: str) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"Missing {split} manifest: {path}")
    df = pd.read_csv(path)
    if "status" in df.columns:
        df = df[df["status"].astype(str).str.lower().eq("success")].copy()
    if "video_id" not in df.columns:
        raise KeyError(f"{path} must contain a video_id column")
    df["video_id"] = df["video_id"].astype(str)
    df = df[~df["video_id"].str.contains("smoke_test", case=False, na=False)].copy()
    if len(df) == 0:
        raise ValueError(f"No usable rows found in {path}")
    return df.reset_index(drop=True)


def resolve_feature_path(row: pd.Series, split: str) -> Path:
    if "feature_path" in row and isinstance(row["feature_path"], str) and row["feature_path"].strip():
        candidate = Path(row["feature_path"])
        if candidate.exists():
            return candidate
    feature_dir = TRAIN_FEATURE_DIR if split == "train" else TEST_FEATURE_DIR
    return feature_dir / f"{row['video_id']}.npz"


def build_label_index(root: Path) -> dict:
    root = Path(root)
    if not root.exists():
        print(f"Warning: GROUND_TRUTH_PATH does not exist yet: {root}")
        print("         Manifest label_path values will still be used when available.")
        return {}
    if root.is_file():
        return {key: root for key in lookup_keys_for_name(root.stem)}

    index = {}
    collisions = set()
    for file_path in root.rglob("*"):
        if not (file_path.is_file() and file_path.suffix.lower() in LABEL_EXTENSIONS):
            continue
        for key in lookup_keys_for_name(file_path.stem):
            if key in index and index[key] != file_path:
                collisions.add(key)
            else:
                index[key] = file_path

    for key in collisions:
        index.pop(key, None)

    if collisions:
        preview = sorted(collisions)[:10]
        print(f"Skipped {len(collisions)} ambiguous label lookup key(s), examples: {preview}")
    if not index:
        print(f"Warning: No label files with extensions {sorted(LABEL_EXTENSIONS)} found under {root}")
    return index


def manifest_label_path(row: pd.Series) -> Path | None:
    if "label_path" not in row:
        return None
    value = row["label_path"]
    if pd.isna(value):
        return None
    text = str(value).strip()
    if not text or text.lower() in {"nan", "none", "null"}:
        return None

    candidates = [Path(text)]
    if "/My Drive/" in text:
        candidates.append(Path(text.replace("/My Drive/", "/MyDrive/")))
    if "/MyDrive/" in text:
        candidates.append(Path(text.replace("/MyDrive/", "/My Drive/")))

    for candidate in candidates:
        if candidate.exists():
            return candidate

    print(f"Warning: manifest label_path does not exist, falling back to label index: {text}")
    return None


def coerce_label_vector(values, source: Path) -> np.ndarray:
    arr = np.asarray(values)
    if arr.dtype == object:
        try:
            arr = np.asarray(arr.tolist())
        except Exception:
            arr = np.asarray([np.asarray(x).squeeze() for x in arr.ravel()], dtype=object)
    arr = np.squeeze(arr)

    if arr.ndim == 0:
        raise ValueError(f"{source}: label array is scalar, expected one value per frame")
    if arr.ndim > 1:
        if 1 in arr.shape:
            arr = arr.reshape(-1)
        else:
            arr = arr.reshape(-1)

    try:
        arr = arr.astype(np.float32)
    except Exception as exc:
        raise ValueError(f"{source}: could not convert labels to numeric values") from exc

    return (arr > 0).astype(np.uint8)


def choose_array_from_npz(npz, video_id: str, source: Path):
    preferred = [video_id, *lookup_keys_for_name(video_id), "labels", "label", "gt", "mask", "frame_mask", "frame_labels", "arr_0"]
    keys = list(npz.files)
    key_map = {key: k for k in keys for key in lookup_keys_for_name(k)}
    for key in preferred:
        normalized = normalize_key(key)
        compact = compact_key(key)
        for candidate_key in [normalized, compact, key]:
            if candidate_key in key_map:
                return npz[key_map[candidate_key]]
    if len(keys) == 1:
        return npz[keys[0]]
    raise KeyError(f"{source}: could not identify label array. Available keys: {keys}")


def choose_array_from_mat(mat_dict: dict, video_id: str, source: Path):
    keys = [k for k in mat_dict.keys() if not k.startswith("__")]
    key_map = {key: k for k in keys for key in lookup_keys_for_name(k)}
    preferred = [video_id, *lookup_keys_for_name(video_id), "labels", "label", "gt", "mask", "frame_mask", "frame_labels"]
    for key in preferred:
        normalized = normalize_key(key)
        compact = compact_key(key)
        for candidate_key in [normalized, compact, key]:
            if candidate_key in key_map:
                return mat_dict[key_map[candidate_key]]
    numeric_keys = [k for k in keys if np.asarray(mat_dict[k]).dtype != object]
    if len(numeric_keys) == 1:
        return mat_dict[numeric_keys[0]]
    if len(keys) == 1:
        return mat_dict[keys[0]]
    raise KeyError(f"{source}: could not identify label array. Available keys: {keys}")


def load_label_file(path: Path, video_id: str, expected_length: int) -> np.ndarray:
    suffix = path.suffix.lower()
    if suffix == ".npy":
        labels = coerce_label_vector(np.load(path, allow_pickle=True), path)
    elif suffix == ".npz":
        labels = coerce_label_vector(choose_array_from_npz(np.load(path, allow_pickle=True), video_id, path), path)
    elif suffix == ".mat":
        labels = coerce_label_vector(choose_array_from_mat(loadmat(path), video_id, path), path)
    elif suffix in {".csv", ".txt"}:
        sep = "," if suffix == ".csv" else r"\s+"
        df = pd.read_csv(path, sep=sep, engine="python")
        # If a one-column label file has no header, pandas treats the first
        # label as the column name. Retry without a header when the row count is
        # off by one or no usable label column is found.
        df_no_header = pd.read_csv(path, sep=sep, engine="python", header=None)
        if len(df) != expected_length and len(df_no_header) == expected_length:
            df = df_no_header
        preferred_cols = ["label", "labels", "gt", "mask", "frame_mask", "anomaly", "is_anomaly"]
        chosen_col = None
        for col in preferred_cols:
            if col in df.columns:
                chosen_col = col
                break
        if chosen_col is None:
            numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
            if not numeric_cols:
                raise ValueError(f"{path}: no numeric label column found")
            # For no-header frame,label files, the final numeric column is
            # usually the label and earlier columns are frame indices/timestamps.
            chosen_col = numeric_cols[-1]
        labels = coerce_label_vector(df[chosen_col].to_numpy(), path)
    else:
        raise ValueError(f"Unsupported label file extension: {path}")

    if labels.shape[0] != expected_length:
        raise ValueError(
            f"{path}: label length {labels.shape[0]} does not match feature frame count {expected_length} for video {video_id}"
        )
    return labels


def find_label_path(video_id: str, label_index: dict, row: pd.Series | None = None) -> Path:
    manifest_path = manifest_label_path(row) if row is not None else None
    if manifest_path is not None:
        return manifest_path

    for key in lookup_keys_for_name(video_id):
        if key in label_index:
            return label_index[key]

    available_preview = sorted(label_index.keys())[:20]
    raise FileNotFoundError(
        f"Missing label file for video_id={video_id}. Looked under {GROUND_TRUTH_PATH} and manifest label_path. "
        f"First available label keys: {available_preview}"
    )



## 7. Load Features, Labels, and Training Statistics

This cell validates the selected dataset's feature NPZ files and mandatory
frame labels before any training starts.



In [ ]:
train_manifest = load_manifest(TRAIN_MANIFEST_PATH, "train")
test_manifest = load_manifest(TEST_MANIFEST_PATH, "test")
label_index = build_label_index(GROUND_TRUTH_PATH)

print(f"Dataset: {DISPLAY_DATASET_NAME} ({DATASET_KEY})")
print(f"Feature root: {FEATURE_ROOT}")
print(f"Ground-truth path: {GROUND_TRUTH_PATH}")
print(f"Train manifest rows: {len(train_manifest)}")
print(f"Test manifest rows: {len(test_manifest)}")
print(f"Indexed label lookup keys: {len(label_index)}")

if not FEATURE_STATS_PATH.exists():
    raise FileNotFoundError(f"Missing training feature statistics: {FEATURE_STATS_PATH}")
stats = np.load(FEATURE_STATS_PATH)
train_mean = stats["mean"].astype(np.float32)
train_std = stats["std"].astype(np.float32)
if train_mean.shape != (INPUT_DIM,) or train_std.shape != (INPUT_DIM,):
    raise ValueError(f"Expected stats vectors of shape ({INPUT_DIM},), got mean={train_mean.shape}, std={train_std.shape}")
if not np.isfinite(train_mean).all() or not np.isfinite(train_std).all():
    raise ValueError("Training feature statistics contain NaN or Inf")
train_std = np.where(train_std < 1e-8, 1.0, train_std).astype(np.float32)


def load_train_features() -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    features_list = []
    video_ids = []
    frame_indices = []
    for _, row in tqdm(train_manifest.iterrows(), total=len(train_manifest), desc="Loading train NPZ"):
        path = resolve_feature_path(row, "train")
        item = validate_feature_npz(path, "train")
        features_list.append(item["features"])
        video_ids.extend([item["video_id"]] * item["num_frames"])
        frame_indices.append(item["frame_indices"])
    features = np.concatenate(features_list, axis=0).astype(np.float32, copy=False)
    return features, np.asarray(video_ids), np.concatenate(frame_indices, axis=0)


def load_test_features_and_labels() -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    features_list = []
    labels_list = []
    video_ids = []
    frame_indices = []
    for _, row in tqdm(test_manifest.iterrows(), total=len(test_manifest), desc="Loading test NPZ + labels"):
        path = resolve_feature_path(row, "test")
        item = validate_feature_npz(path, "test")
        label_path = find_label_path(item["video_id"], label_index, row=row)
        labels = load_label_file(label_path, item["video_id"], item["num_frames"])
        features_list.append(item["features"])
        labels_list.append(labels)
        video_ids.extend([item["video_id"]] * item["num_frames"])
        frame_indices.append(item["frame_indices"])
    features = np.concatenate(features_list, axis=0).astype(np.float32, copy=False)
    labels = np.concatenate(labels_list, axis=0).astype(np.uint8, copy=False)
    return features, labels, np.asarray(video_ids), np.concatenate(frame_indices, axis=0)


train_features_raw, train_video_ids, train_frame_indices = load_train_features()
test_features_raw, test_labels, test_video_ids, test_frame_indices = load_test_features_and_labels()

train_features = ((train_features_raw - train_mean) / (train_std + 1e-8)).astype(np.float32)
test_features = ((test_features_raw - train_mean) / (train_std + 1e-8)).astype(np.float32)

del train_features_raw, test_features_raw

if not np.isfinite(train_features).all():
    raise ValueError("Standardized train features contain NaN or Inf")
if not np.isfinite(test_features).all():
    raise ValueError("Standardized test features contain NaN or Inf")
if len(np.unique(test_labels)) < 2:
    raise ValueError("All concatenated test labels have one class only; frame-level AUC is undefined")

summary = {
    "dataset_key": DATASET_KEY,
    "dataset_name": DISPLAY_DATASET_NAME,
    "num_train_frames": int(train_features.shape[0]),
    "num_test_frames": int(test_features.shape[0]),
    "num_train_videos": int(len(np.unique(train_video_ids))),
    "num_test_videos": int(len(np.unique(test_video_ids))),
    "test_normal_frames": int((test_labels == 0).sum()),
    "test_anomaly_frames": int((test_labels == 1).sum()),
    "feature_dim": int(train_features.shape[1]),
}
with open(OUTPUT_ROOT / "data_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

print(json.dumps(summary, indent=2))


## 8. Create the Official MULDE Model and DataLoader

The network input dimension is `1152 + 1`: the Hiera-L feature vector plus the
scalar noise-level conditioning value.



In [ ]:
model = ScoreOrLogDensityNetwork(
    MLPs(
        input_dim=INPUT_DIM + 1,
        output_dim=1,
        units=UNITS,
        dropout=None,
        layernorm=False,
    ),
    score_network=False,
).to(DEVICE)

optimizer = optim.Adam(model.parameters(), lr=LR, betas=ADAM_BETAS)

# CHANGED: Added StepLR learning rate scheduler based on original paper implementation
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=50, gamma=0.9)

start_epoch = 1
if RESUME_CHECKPOINT is not None:
    # Set weights_only=False to support loading legacy checkpoints containing numpy arrays
    checkpoint = torch.load(RESUME_CHECKPOINT, map_location=DEVICE, weights_only=False)
    model.load_state_dict(checkpoint["model_state_dict"])
    optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
    start_epoch = int(checkpoint["epoch"]) + 1
    print(f"Resumed from {RESUME_CHECKPOINT}; next epoch is {start_epoch}")

train_tensor = torch.from_numpy(train_features)
train_loader = DataLoader(
    TensorDataset(train_tensor),
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    drop_last=False,
)

num_params = sum(p.numel() for p in model.parameters())
print(model)
print(f"Trainable parameters: {num_params:,}")
print(f"Training batches per epoch: {len(train_loader)}")

## 9. Training Loop: Multiscale Denoising Score Matching

This implements the author's DSM logic from `main.py`:
- sample `sigma` log-uniformly on `[1e-3, 1.0]`
- add Gaussian noise to standardized features
- compute the score as the gradient of `-f_theta(x_noisy, sigma)` using
  `ScoreOrLogDensityNetwork.score`
- use `lambda(sigma)=sigma^2`
- add the paper regularizer `beta * f_theta(x_clean, sigma)^2 / 2`

Losses are printed and appended to `training_log.csv`. Checkpoints are saved
every 50 epochs and at the end.



In [ ]:
TRAINING_LOG_PATH = LOG_DIR / "training_log.csv"
PERIODIC_EVAL_LOG_PATH = LOG_DIR / "periodic_eval_log.csv"
PERIODIC_EVAL_JSONL_PATH = LOG_DIR / "periodic_eval_metrics.jsonl"

SIGMA_LEVELS = np.linspace(SIGMA_LOW, SIGMA_HIGH, L, dtype=np.float32)
print("Evaluation sigmas:", SIGMA_LEVELS)


def append_training_log(row: dict) -> None:
    df = pd.DataFrame([row])
    df.to_csv(TRAINING_LOG_PATH, mode="a", header=not TRAINING_LOG_PATH.exists(), index=False)


def append_periodic_eval_log(rows: list[dict]) -> None:
    if not rows:
        return
    df = pd.DataFrame(rows)
    df.to_csv(PERIODIC_EVAL_LOG_PATH, mode="a", header=not PERIODIC_EVAL_LOG_PATH.exists(), index=False)
    with open(PERIODIC_EVAL_JSONL_PATH, "a", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row) + "\n")


def save_checkpoint(epoch: int, final: bool = False) -> Path:
    name = "mulde_final.pt" if final else f"mulde_epoch_{epoch:04d}.pt"
    path = CHECKPOINT_DIR / name
    torch.save(
        {
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "config": CONFIG,
            "train_mean": train_mean,
            "train_std": train_std,
        },
        path,
    )
    return path


def sample_log_uniform_sigma(batch_size: int, device: torch.device) -> torch.Tensor:
    log_low = math.log(SIGMA_LOW)
    log_high = math.log(SIGMA_HIGH)
    return torch.exp(torch.empty(batch_size, 1, device=device).uniform_(log_low, log_high))


def train_one_epoch(epoch: int) -> dict:
    model.train()
    loss_values = []
    dsm_values = []
    reg_values = []
    score_norm_values = []
    log_density_values = []

    progress = tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS}", leave=False)
    for (x_cpu,) in progress:
        x = x_cpu.to(DEVICE, non_blocking=True)
        sigma = sample_log_uniform_sigma(x.shape[0], DEVICE)
        noise = torch.randn_like(x, device=DEVICE) * sigma
        x_noisy = x + noise

        # Official MULDE score convention: score() returns grad(-f_theta).
        score, log_density_noisy = model.score(
            torch.cat([x_noisy, sigma], dim=1),
            return_log_density=True,
        )

        score_x = score[:, :-1]  # exclude gradient w.r.t. sigma conditioning dimension
        dsm_per_sample = torch.linalg.vector_norm(score_x + noise / (sigma ** 2), dim=1) ** 2
        lambda_factor = (sigma ** 2).reshape(-1)
        loss_dsm = (lambda_factor * dsm_per_sample).mean() / 2.0

        # Same regularizer as the paper and author code, computed by direct
        # forward pass to avoid an unused clean-input score gradient.
        log_density_clean = model(torch.cat([x, sigma], dim=1))
        loss_reg = BETA * (log_density_clean.reshape(-1) ** 2).mean() / 2.0
        loss = loss_dsm + loss_reg

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()

        loss_values.append(float(loss.detach().cpu()))
        dsm_values.append(float(loss_dsm.detach().cpu()))
        reg_values.append(float(loss_reg.detach().cpu()))
        score_norm_values.append(float(torch.linalg.vector_norm(score_x.detach(), dim=1).pow(2).mean().cpu()))
        log_density_values.append(float(log_density_noisy.detach().mean().cpu()))
        progress.set_postfix(loss=np.mean(loss_values), dsm=np.mean(dsm_values), reg=np.mean(reg_values))

    row = {
        "epoch": epoch,
        "loss_total": float(np.mean(loss_values)),
        "loss_dsm": float(np.mean(dsm_values)),
        "loss_regularizer": float(np.mean(reg_values)),
        "score_norm_mean": float(np.mean(score_norm_values)),
        "log_density_noisy_mean": float(np.mean(log_density_values)),
        "lr": float(optimizer.param_groups[0]["lr"]),
        "timestamp": datetime.datetime.now().isoformat(),
    }
    append_training_log(row)
    return row


def compute_multiscale_log_density_for_eval(
    features_std: np.ndarray,
    name: str,
    batch_size: int = EVAL_BATCH_SIZE,
) -> np.ndarray:
    model.eval()
    n = features_std.shape[0]
    out = np.empty((n, len(SIGMA_LEVELS)), dtype=np.float32)

    with torch.no_grad():
        for start in tqdm(range(0, n, batch_size), desc=f"Periodic scoring {name}", leave=False):
            end = min(start + batch_size, n)
            x = torch.from_numpy(features_std[start:end]).to(DEVICE, non_blocking=True)
            cols = []
            for sigma_value in SIGMA_LEVELS:
                sigma_col = torch.full((x.shape[0], 1), float(sigma_value), device=DEVICE)
                log_density = model(torch.cat([x, sigma_col], dim=1)).reshape(-1)
                cols.append(log_density.detach().cpu().numpy().astype(np.float32))
            out[start:end] = np.stack(cols, axis=1)
    return out


def apply_optional_video_smoothing_for_eval(df: pd.DataFrame, score_col: str) -> str:
    eval_col = f"{score_col}_eval"
    if SMOOTH_SIGMA_FRAMES is None:
        df[eval_col] = df[score_col].to_numpy(dtype=np.float32)
        return eval_col

    smoothed = np.empty(len(df), dtype=np.float32)
    for _, indices in df.groupby("video_id", sort=False).groups.items():
        idx = np.asarray(list(indices))
        values = df.loc[idx, score_col].to_numpy(dtype=np.float32)
        smoothed[idx] = gaussian_filter1d(values, sigma=float(SMOOTH_SIGMA_FRAMES), mode="nearest")
    df[eval_col] = smoothed
    return eval_col


def safe_auc_for_eval(labels: np.ndarray, scores: np.ndarray):
    if len(np.unique(labels)) < 2:
        return None
    return float(roc_auc_score(labels, scores))


def compute_micro_macro_auc_for_eval(df: pd.DataFrame, score_col: str) -> dict:
    y = df["label"].to_numpy(dtype=np.uint8)
    s = df[score_col].to_numpy(dtype=np.float32)
    micro_auc = safe_auc_for_eval(y, s)

    video_aucs = []
    skipped = []
    for video_id, sub in df.groupby("video_id", sort=False):
        auc = safe_auc_for_eval(sub["label"].to_numpy(dtype=np.uint8), sub[score_col].to_numpy(dtype=np.float32))
        if auc is None:
            skipped.append(str(video_id))
        else:
            video_aucs.append(auc)

    macro_auc = float(np.mean(video_aucs)) if video_aucs else None
    return {
        "micro_auc": micro_auc,
        "macro_auc": macro_auc,
        "num_macro_videos": int(len(video_aucs)),
        "skipped_macro_videos_one_class": skipped,
    }


def evaluate_split_with_gmm(
    split: str,
    epoch: int,
    components: int,
    gmm: GaussianMixture,
    log_density: np.ndarray,
    video_ids: np.ndarray,
    frame_indices: np.ndarray,
    labels: np.ndarray,
) -> dict:
    score_col = f"gmm_{components}_nll"
    df = pd.DataFrame(
        {
            "video_id": video_ids,
            "frame_index": frame_indices.astype(np.int64),
            "label": labels.astype(np.uint8),
            score_col: (-gmm.score_samples(log_density)).astype(np.float32),
        }
    )
    eval_score_col = apply_optional_video_smoothing_for_eval(df, score_col)
    auc_info = compute_micro_macro_auc_for_eval(df, eval_score_col)
    scores = df[eval_score_col].to_numpy(dtype=np.float32)

    return {
        "epoch": int(epoch),
        "split": split,
        "components": int(components),
        "score_col": eval_score_col,
        "micro_auc": auc_info["micro_auc"],
        "macro_auc": auc_info["macro_auc"],
        "num_macro_videos": auc_info["num_macro_videos"],
        "skipped_macro_videos_one_class": "|".join(auc_info["skipped_macro_videos_one_class"]),
        "nll_mean": float(np.mean(scores)),
        "nll_std": float(np.std(scores)),
        "nll_min": float(np.min(scores)),
        "nll_max": float(np.max(scores)),
        "smooth_sigma_frames": SMOOTH_SIGMA_FRAMES,
        "lr_used_this_epoch": None,
        "lr_after_scheduler": float(optimizer.param_groups[0]["lr"]),
        "timestamp": datetime.datetime.now().isoformat(),
    }


def format_auc(value) -> str:
    return "NA" if value is None else f"{value:.6f}"


def run_periodic_gmm_evaluation(epoch: int, lr_used_this_epoch: float) -> list[dict]:
    eval_start = time.time()
    print(f"\n--- Periodic GMM evaluation at epoch {epoch} ---")
    train_log_density_eval = compute_multiscale_log_density_for_eval(train_features, f"train epoch {epoch}")
    test_log_density_eval = compute_multiscale_log_density_for_eval(test_features, f"test epoch {epoch}")

    # MULDE training splits are normal-only, so train AUC is undefined unless
    # a future dataset supplies train labels with both classes.
    train_eval_labels = np.zeros(train_log_density_eval.shape[0], dtype=np.uint8)

    rows = []
    best_test_row = None
    for components in GMM_COMPONENTS:
        gmm = GaussianMixture(
            n_components=components,
            covariance_type=GMM_COVARIANCE,
            random_state=SEED,
        )
        gmm.fit(train_log_density_eval)

        train_row = evaluate_split_with_gmm(
            "train",
            epoch,
            components,
            gmm,
            train_log_density_eval,
            train_video_ids,
            train_frame_indices,
            train_eval_labels,
        )
        test_row = evaluate_split_with_gmm(
            "test",
            epoch,
            components,
            gmm,
            test_log_density_eval,
            test_video_ids,
            test_frame_indices,
            test_labels,
        )
        train_row["lr_used_this_epoch"] = float(lr_used_this_epoch)
        test_row["lr_used_this_epoch"] = float(lr_used_this_epoch)
        rows.extend([train_row, test_row])

        if test_row["micro_auc"] is not None and (
            best_test_row is None or test_row["micro_auc"] > best_test_row["micro_auc"]
        ):
            best_test_row = test_row

        print(
            f"Epoch {epoch:04d} | GMM({components}) | "
            f"train micro={format_auc(train_row['micro_auc'])} macro={format_auc(train_row['macro_auc'])} | "
            f"test micro={format_auc(test_row['micro_auc'])} macro={format_auc(test_row['macro_auc'])}"
        )

    elapsed_min = (time.time() - eval_start) / 60.0
    for row in rows:
        row["eval_elapsed_min"] = float(elapsed_min)
    append_periodic_eval_log(rows)

    if best_test_row is not None:
        print(
            f"Best periodic test AUC at epoch {epoch}: "
            f"GMM({best_test_row['components']}) micro={best_test_row['micro_auc']:.6f} "
            f"macro={format_auc(best_test_row['macro_auc'])}"
        )
    print(f"Saved periodic eval log: {PERIODIC_EVAL_LOG_PATH}")
    return rows


training_start = time.time()
for epoch in range(start_epoch, EPOCHS + 1):
    row = train_one_epoch(epoch)

    scheduler.step()

    print(
        f"Epoch {epoch:04d}/{EPOCHS} | "
        f"loss={row['loss_total']:.6f} | dsm={row['loss_dsm']:.6f} | reg={row['loss_regularizer']:.6f}"
    )

    if PERIODIC_EVAL_EVERY and epoch % PERIODIC_EVAL_EVERY == 0:
        run_periodic_gmm_evaluation(epoch, lr_used_this_epoch=row["lr"])

    if CHECKPOINT_EVERY and epoch % CHECKPOINT_EVERY == 0:
        ckpt_path = save_checkpoint(epoch, final=False)
        print(f"Saved checkpoint: {ckpt_path}")

final_checkpoint = save_checkpoint(EPOCHS, final=True)
state_dict_path = CHECKPOINT_DIR / "mulde_final_state_dict.pt"
model.save(str(state_dict_path))

elapsed_min = (time.time() - training_start) / 60.0
print(f"Training complete in {elapsed_min:.2f} minutes")
print(f"Final checkpoint: {final_checkpoint}")
print(f"Final state dict via official model.save(): {state_dict_path}")
print(f"Periodic eval CSV: {PERIODIC_EVAL_LOG_PATH}")
print(f"Periodic eval JSONL: {PERIODIC_EVAL_JSONL_PATH}")


## 10. Compute Final Multiscale Log-Density Vectors

During training, this notebook now evaluates train/test GMM metrics every
`PERIODIC_EVAL_EVERY` epochs. After training finishes, this cell recomputes the
final multiscale log-density vectors and saves them as artifacts for the final
GMM evaluation/export stage.



In [ ]:
SIGMA_LEVELS = np.linspace(SIGMA_LOW, SIGMA_HIGH, L, dtype=np.float32)
print("Evaluation sigmas:", SIGMA_LEVELS)


def compute_multiscale_log_density(features_std: np.ndarray, name: str, batch_size: int = BATCH_SIZE) -> np.ndarray:
    model.eval()
    n = features_std.shape[0]
    out = np.empty((n, len(SIGMA_LEVELS)), dtype=np.float32)

    with torch.no_grad():
        for start in tqdm(range(0, n, batch_size), desc=f"Scoring {name}"):
            end = min(start + batch_size, n)
            x = torch.from_numpy(features_std[start:end]).to(DEVICE, non_blocking=True)
            cols = []
            for sigma_value in SIGMA_LEVELS:
                sigma_col = torch.full((x.shape[0], 1), float(sigma_value), device=DEVICE)
                log_density = model(torch.cat([x, sigma_col], dim=1)).reshape(-1)
                cols.append(log_density.detach().cpu().numpy().astype(np.float32))
            out[start:end] = np.stack(cols, axis=1)
    return out


train_log_density = compute_multiscale_log_density(train_features, "train")
test_log_density = compute_multiscale_log_density(test_features, "test")

train_ld_path = ARTIFACT_DIR / "train_multiscale_log_density.npz"
test_ld_path = ARTIFACT_DIR / "test_multiscale_log_density.npz"

np.savez_compressed(
    train_ld_path,
    log_density=train_log_density,
    sigmas=SIGMA_LEVELS,
    video_ids=train_video_ids,
    frame_indices=train_frame_indices,
)
np.savez_compressed(
    test_ld_path,
    log_density=test_log_density,
    sigmas=SIGMA_LEVELS,
    labels=test_labels,
    video_ids=test_video_ids,
    frame_indices=test_frame_indices,
)

print(f"Saved train log-density vectors: {train_ld_path}")
print(f"Saved test log-density vectors: {test_ld_path}")
print(f"Train log-density shape: {train_log_density.shape}")
print(f"Test log-density shape: {test_log_density.shape}")



## 11. Fit GMMs and Evaluate Frame-Level AUC

This cell fits 1-, 3-, and 5-component full-covariance GMMs on training
multiscale log-density vectors, then uses negative GMM log-likelihood as the
frame anomaly score.

Raw scores are used by default. If you later set `SMOOTH_SIGMA_FRAMES` to a
numeric value, smoothing is applied independently within each test video before
AUC computation.



In [ ]:
# Smoothing is configured in Step 4 and used by both periodic and final evaluation.
print(f"SMOOTH_SIGMA_FRAMES = {SMOOTH_SIGMA_FRAMES}")


In [ ]:
def apply_optional_video_smoothing(df: pd.DataFrame, score_col: str) -> str:
    eval_col = f"{score_col}_eval"
    if SMOOTH_SIGMA_FRAMES is None:
        df[eval_col] = df[score_col].to_numpy(dtype=np.float32)
        return eval_col

    smoothed = np.empty(len(df), dtype=np.float32)
    for _, indices in df.groupby("video_id", sort=False).groups.items():
        idx = np.asarray(list(indices))
        values = df.loc[idx, score_col].to_numpy(dtype=np.float32)
        smoothed[idx] = gaussian_filter1d(values, sigma=float(SMOOTH_SIGMA_FRAMES), mode="nearest")
    df[eval_col] = smoothed
    return eval_col


def safe_auc(labels: np.ndarray, scores: np.ndarray):
    if len(np.unique(labels)) < 2:
        return None
    return float(roc_auc_score(labels, scores))


def compute_micro_macro_auc(df: pd.DataFrame, score_col: str) -> dict:
    y = df["label"].to_numpy(dtype=np.uint8)
    s = df[score_col].to_numpy(dtype=np.float32)
    micro_auc = safe_auc(y, s)

    per_video_rows = []
    video_aucs = []
    skipped = []
    for video_id, sub in df.groupby("video_id", sort=False):
        auc = safe_auc(sub["label"].to_numpy(dtype=np.uint8), sub[score_col].to_numpy(dtype=np.float32))
        row = {
            "video_id": video_id,
            "score_col": score_col,
            "num_frames": int(len(sub)),
            "num_anomaly_frames": int(sub["label"].sum()),
            "auc": auc,
        }
        per_video_rows.append(row)
        if auc is None:
            skipped.append(video_id)
        else:
            video_aucs.append(auc)

    macro_auc = float(np.mean(video_aucs)) if video_aucs else None
    return {
        "micro_auc": micro_auc,
        "macro_auc": macro_auc,
        "num_macro_videos": int(len(video_aucs)),
        "skipped_macro_videos_one_class": skipped,
        "per_video_rows": per_video_rows,
    }


scores_df = pd.DataFrame(
    {
        "video_id": test_video_ids,
        "frame_index": test_frame_indices.astype(np.int64),
        "label": test_labels.astype(np.uint8),
    }
)

for j, sigma_value in enumerate(SIGMA_LEVELS, start=1):
    scores_df[f"log_density_sigma_{j:02d}_{sigma_value:.5f}"] = test_log_density[:, j - 1]

metrics = {
    "config": CONFIG,
    "smoothing": {
        "smooth_sigma_frames": SMOOTH_SIGMA_FRAMES,
        "note": "Raw scores are used when smooth_sigma_frames is None.",
    },
    "gmm": {},
}
all_per_video_rows = []

for components in GMM_COMPONENTS:
    print(f"Fitting GMM with {components} component(s) ...")
    gmm = GaussianMixture(
        n_components=components,
        covariance_type=GMM_COVARIANCE,
        random_state=SEED,
    )
    gmm.fit(train_log_density)

    gmm_path = ARTIFACT_DIR / f"gmm_components_{components}.joblib"
    joblib.dump(gmm, gmm_path)

    raw_score_col = f"gmm_{components}_nll"
    scores_df[raw_score_col] = (-gmm.score_samples(test_log_density)).astype(np.float32)
    eval_score_col = apply_optional_video_smoothing(scores_df, raw_score_col)

    auc_info = compute_micro_macro_auc(scores_df, eval_score_col)
    metrics["gmm"][str(components)] = {
        "artifact": str(gmm_path),
        "raw_score_col": raw_score_col,
        "eval_score_col": eval_score_col,
        "micro_auc": auc_info["micro_auc"],
        "macro_auc": auc_info["macro_auc"],
        "num_macro_videos": auc_info["num_macro_videos"],
        "skipped_macro_videos_one_class": auc_info["skipped_macro_videos_one_class"],
    }
    all_per_video_rows.extend(auc_info["per_video_rows"])

    print(
        f"GMM({components}) | micro AUC={auc_info['micro_auc']:.6f} | "
        f"macro AUC={auc_info['macro_auc']:.6f} | saved={gmm_path}"
    )

scores_path = ARTIFACT_DIR / "test_frame_scores.csv"
metrics_path = ARTIFACT_DIR / "metrics.json"
per_video_metrics_path = ARTIFACT_DIR / "per_video_metrics.csv"

scores_df.to_csv(scores_path, index=False)
pd.DataFrame(all_per_video_rows).to_csv(per_video_metrics_path, index=False)
with open(metrics_path, "w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2)

print(f"Saved frame scores: {scores_path}")
print(f"Saved per-video metrics: {per_video_metrics_path}")
print(f"Saved metrics JSON: {metrics_path}")
print(json.dumps(metrics["gmm"], indent=2))



## 12. Final Artifact Summary

Run this after evaluation to list the files written to Google Drive.



## 12. Export Frame-Level Scores for STG-NF + MULDE Ensemble

This cell exports the raw frame-level anomaly scores from the best GMM so they
can be fused with STG-NF scores in a downstream ensemble notebook.
The exported file is a dictionary keyed by `video_id` whose values are parallel
arrays of `(frame_index, anomaly_score)` tuples. This keeps the per-video
structure required by the ensemble handoff plan.

The export includes the selected dataset key so downstream PRISM code can keep
ShanghaiTech and Avenue artifacts separate.



In [ ]:
import pickle
from collections import OrderedDict

# Pick the best GMM component count from the metrics computed above. Falls
# back to the largest configured component count when no AUC is unavailable.
available_gmms = sorted(metrics["gmm"].keys(), key=lambda k: int(k))
best_components = None
best_components_auc = -1.0
for k in available_gmms:
    auc = metrics["gmm"][k].get("micro_auc")
    if auc is not None and auc > best_components_auc:
        best_components_auc = float(auc)
        best_components = int(k)
if best_components is None:
    best_components = int(available_gmms[-1])
best_score_col = f"gmm_{best_components}_nll"
print(f"Exporting MULDE scores from GMM components={best_components} "
      f"(single-model Micro AUC={best_components_auc:.6f})")

# Build the {video_id: {frame_index: anomaly_score}} mapping required by
# the PRISM script. An OrderedDict preserves the manifest iteration order, which
# makes the resulting PKL easier to inspect.
mulde_scores = OrderedDict()
for video_id, sub in scores_df.groupby("video_id", sort=False):
    frame_index = sub["frame_index"].to_numpy(dtype=np.int64)
    score = sub[best_score_col].to_numpy(dtype=np.float32)
    labels = sub["label"].to_numpy(dtype=np.uint8)
    mulde_scores[str(video_id)] = {
        "frame_indices": frame_index,
        "anomaly_scores": score,
        "labels": labels,
    }

mulde_scores_path = ARTIFACT_DIR / "mulde_scores.pkl"
with open(mulde_scores_path, "wb") as f:
    pickle.dump(
        {
            "dataset_key": DATASET_KEY,
            "dataset_name": DISPLAY_DATASET_NAME,
            "scores_by_video": mulde_scores,
            "best_components": best_components,
            "best_micro_auc": best_components_auc,
            "smoothing": {"smooth_sigma_frames": SMOOTH_SIGMA_FRAMES},
            "config": CONFIG,
        },
        f,
    )

print(f"Saved MULDE frame scores: {mulde_scores_path}")
print(f"Number of test videos: {len(mulde_scores)}")
total_frames = sum(v["frame_indices"].shape[0] for v in mulde_scores.values())
print(f"Total test frames exported: {total_frames}")


## 13. Final Artifact Summary

Run this after evaluation to list the files written to Google Drive.



In [ ]:
print(f"Run output root: {OUTPUT_ROOT}")
for path in sorted(OUTPUT_ROOT.rglob("*")):
    if path.is_file():
        size_mb = path.stat().st_size / (1024 * 1024)
        print(f"{size_mb:8.2f} MB  {path}")
